# 04 · K-Means clustering

Fit K-Means on reviewed features from the shared preprocessing pipeline. The active dataset is discovered from `data/raw`; this notebook does not assume survey column names.

In [ ]:
import sys
from pathlib import Path

project = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project))

from src.data_loader import discover_datasets, load_dataset
from src.feature_engineering import default_clustering_features
from src.data_cleaning import preprocess_data
from src.clustering import fit_kmeans, elbow_scores
from src.profiling import cluster_profile

files = discover_datasets(project)
assert files, "Add a supported dataset under data/raw before running this notebook."
source = next((path for path in files if path.suffix.lower() == ".dta"), next((path for path in files if path.suffix.lower() == ".csv"), files[0]))
info = load_dataset(source)
frame = info.frame
features = default_clustering_features(frame)
print("Source:", source.name)
print("Shape:", frame.shape)
print("Candidate features:", features)
assert len(frame) >= 3, "At least three records are required for this K-Means example."
assert features, "No eligible clustering features were detected."

In [ ]:
matrix, preprocessor, selected_features = preprocess_data(frame, features)
k = min(4, len(frame) - 1)
result = fit_kmeans(matrix, n_clusters=k)
profile, segment_names = cluster_profile(frame, result.labels)
print("Selected features:", selected_features)
print("Transformed feature matrix:", matrix.shape)
print("Evaluation:", result.metrics)
print("Inertia:", result.inertia)
print("Cluster sizes:")
print(frame.assign(cluster=result.labels)["cluster"].value_counts().sort_index())
print("Cluster profile:")
display(profile)
print("Data-derived descriptive labels:")
display(segment_names)